<a href="https://colab.research.google.com/github/josegabrielgc09/COMP-1818-CHATBOT/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets scikit-learn torch numpy


In [2]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

def run_experiment():
    print("1. Loading CLINC150 (OOS) Dataset...")
    dataset = load_dataset("clinc/clinc_oos", "plus")

    train_texts = dataset['train']['text']
    train_labels = dataset['train']['intent']
    test_texts = dataset['test']['text']
    test_labels = dataset['test']['intent']

    print("\n2. Training Baseline Model (TF-IDF + SVM)...")
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_tfidf = vectorizer.fit_transform(train_texts)
    X_test_tfidf = vectorizer.transform(test_texts)

    svm_classifier = SVC(kernel='linear', C=1.0)
    svm_classifier.fit(X_train_tfidf, train_labels)

    svm_predictions = svm_classifier.predict(X_test_tfidf)
    svm_acc = accuracy_score(test_labels, svm_predictions)
    print(f"--> Baseline SVM In-Scope/OOS Accuracy: {svm_acc:.4f}")

    print("\n3. Tokenizing and Fine-Tuning DistilBERT...")
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(batch):
        tokenized = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)
        tokenized["labels"] = batch["intent"]
        return tokenized

    tokenized_data = dataset.map(tokenize_fn, batched=True)

    num_classes = len(set(train_labels))
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes
    )

    training_args = TrainingArguments(
        output_dir="./model_output",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="no"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": accuracy_score(labels, preds)}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_data["train"],
        eval_dataset=tokenized_data["test"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_metrics = trainer.evaluate()
    print(f"\n--> Proposed DistilBERT Accuracy: {eval_metrics['eval_accuracy']:.4f}")

if __name__ == "__main__":
    run_experiment()


1. Loading CLINC150 (OOS) Dataset...


README.md:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

plus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  312kB            

plus/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plus/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 77.8kB            

plus/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (16 outputs left out)

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,3.287493,3.132828,0.685818
2,1.902258,1.975748,0.783455
3,1.454585,1.653895,0.807455


Training Loss,Validation Loss,Epoch,Accuracy
1.454585,1.653895,3,0.807455



--> Proposed DistilBERT Accuracy: 0.8075
